# Implied-Volatility Uncertainty Analysis

This notebook measures how bid-ask quote uncertainty becomes implied-volatility uncertainty. Instead of treating implied volatility as one exact number, the analysis keeps separate IV estimates from bid, mid, and ask prices.

## IV uncertainty measures

The main uncertainty interval is:

\[
IV_{\text{range}} = IV_{\text{ask}} - IV_{\text{bid}}
\]

The relative range scales that interval by the mid implied volatility:

\[
IV_{\text{relative range}} = \frac{IV_{\text{range}}}{IV_{\text{mid}}}
\]

A wider interval means the quoted option price supports a less precise implied-volatility estimate.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.iv_uncertainty import (
    iv_uncertainty_by_expiry,
    iv_uncertainty_by_liquidity,
    iv_uncertainty_by_moneyness,
    iv_uncertainty_summary,
    plot_iv_bid_mid_ask_smile,
    plot_iv_uncertainty_by_expiry,
    plot_iv_uncertainty_by_moneyness,
    plot_iv_uncertainty_heatmap,
    prepare_iv_uncertainty_dataset,
)

## Load IV results

The input file should contain cleaned quote fields and the columns `IV_bid`, `IV_mid`, and `IV_ask`.

In [ ]:
iv_input_path = project_root / "data" / "processed" / "iv_uncertainty_results.csv"

if not iv_input_path.exists():
    raise FileNotFoundError("IV uncertainty input file not found.")

iv_quotes = pd.read_csv(iv_input_path)
iv_quotes.head(10)

## Prepare analysis fields

The dataset is grouped by moneyness, expiry, option type, and liquidity bucket.

In [ ]:
iv_analysis = prepare_iv_uncertainty_dataset(iv_quotes)

iv_analysis_columns = [
    column
    for column in [
        "contractSymbol",
        "expiration",
        "option_type",
        "strike",
        "log_moneyness",
        "moneyness_bucket",
        "days_to_expiry",
        "expiry_bucket",
        "spread_pct",
        "liquidity_bucket",
        "IV_bid",
        "IV_mid",
        "IV_ask",
        "IV_range",
        "IV_relative_range",
    ]
    if column in iv_analysis.columns
]

iv_analysis[iv_analysis_columns].head(15)

In [ ]:
iv_analysis_path = project_root / "data" / "processed" / "iv_uncertainty_analysis.csv"
iv_analysis.to_csv(iv_analysis_path, index=False)

iv_analysis_path

## IV uncertainty summary

The summary table shows where IV intervals are widest after grouping by option type, expiry bucket, moneyness bucket, and liquidity bucket.

In [ ]:
summary = iv_uncertainty_summary(iv_analysis)
summary.head(20)

In [ ]:
summary_path = project_root / "data" / "processed" / "iv_uncertainty_summary.csv"
summary.to_csv(summary_path, index=False)

summary_path

## Grouped views

In [ ]:
expiry_summary = iv_uncertainty_by_expiry(iv_analysis)
expiry_summary.head(20)

In [ ]:
moneyness_summary = iv_uncertainty_by_moneyness(iv_analysis)
moneyness_summary.head(20)

In [ ]:
liquidity_summary = iv_uncertainty_by_liquidity(iv_analysis)
liquidity_summary.head(20)

## Figures

In [ ]:
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

plot_iv_bid_mid_ask_smile(
    iv_analysis,
    figures_dir / "iv_bid_mid_ask_smile.png",
)

plot_iv_uncertainty_heatmap(
    iv_analysis,
    figures_dir / "iv_uncertainty_heatmap.png",
)

plot_iv_uncertainty_by_expiry(
    iv_analysis,
    figures_dir / "iv_uncertainty_by_expiry.png",
)

plot_iv_uncertainty_by_moneyness(
    iv_analysis,
    figures_dir / "iv_uncertainty_by_moneyness.png",
)

## Interpretation notes

IV uncertainty is usually larger when the option quote has a wide bid-ask spread, low liquidity, short time to expiry, or very far-from-the-money strikes. These effects are important because the same market quote can produce a range of plausible implied volatilities instead of one precise value.